# Setup

In [ ]:
pip install pyspark

In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DateType, TimestampType
from pyspark.sql.functions import (
    col, size, lit, explode,
    concat, concat_ws, substring,
    datediff, date_add, date_sub,
    year, month, dayofmonth, dayofweek, dayofyear, weekofyear,
    hour, minute, second,
    count, min, max, avg, sum, udf, when
)
from datetime import datetime

spark = SparkSession.builder.appName("a").getOrCreate()

# 2. Początki z PySpark

## 2.1. Tworzenie DataFrame

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder.appName('spark').getOrCreate()

In [ ]:
df = spark.createDataFrame(
    [
        ("Marcelina", "Tetlak", 32),
        ("Anna", "Radomska", 42),

    ],
    ['first', 'last', 'age']
)

In [ ]:
df.show()

## 2.2. Czytanie danych z .csv

In [ ]:
csv1 = spark.read.format('csv').load('data/best_selling_books.csv')
csv1.show()

In [ ]:
csv2 = spark.read.format('csv').load('data/country-codes.csv')
csv2.show()

## 2.3. Konfiguracja odczytu .csv

In [ ]:
csv3 = (
    spark.read
    .format('csv')
    .options(header=True, sep=",")
    .load('data/best_selling_books.csv')
)
csv3.show()

In [ ]:
csv4 = (
    spark.read
    .format('csv')
    .options(header=False, sep=";")
    .load('data/country-codes.csv')
)
csv4.show()

# 3. Schematy

## 3.1. Wyświetlanie schematu DF

In [ ]:
df = spark.read.csv("data/Games.csv", header=True, quote="\"") # "Atari, Inc. (Windows)"
df.show()

In [ ]:
df.printSchema()

## 3.2. Tworzenie schematu

In [ ]:
spark.read.csv("data/best_selling_books.csv", header=True).printSchema()

## 3.3. Implementacja schematu

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

In [ ]:
schema = StructType(
    [
        StructField("Book", StringType(), False),
        StructField("Authors", StringType(), False),
        StructField("Original Language", StringType(), False),
        StructField("First published", IntegerType(), False),
        StructField("Sales", DoubleType(), False),
        StructField("Genre", StringType(), False)
    ]
)

In [ ]:
print(schema)

In [ ]:
df = spark.read.csv("data/best_selling_books.csv", header=True, schema=schema)
df.printSchema()

In [ ]:
csv1 = spark.read.format("csv").schema(schema).load("data/best_selling_books.csv")
csv1.printSchema()

In [ ]:
csv1 = spark.read.format("csv").schema(schema).load("data/best_selling_books.csv")
csv1.show()

# 4. Selekcja danych

## 4.1. Wyświetlanie wybranych kolumn

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import col

In [ ]:
spark = SparkSession.builder.appName("spark").getOrCreate()

In [ ]:
games_schema = StructType(
    [
        StructField("Name", StringType(), False),
        StructField("Sales", DoubleType(), False),
        StructField("Series", StringType(), True),
        StructField("Release", StringType(), False),
        StructField("Genre", StringType(), False),
        StructField("Developer", StringType(), False),
        StructField("Publisher", StringType(), False)
    ]
)

In [ ]:
df = spark.read.csv("data/Games.csv", header=True, schema=games_schema)
df.show()

In [ ]:
df.select (col('name'), col('sales'), col('developer')).show()

In [ ]:
df.select('name', 'sales', 'developer').show()

## 4.2. Sortowanie danych

In [ ]:
df.orderBy( col("developer").asc(), col("sales").desc() ).show()

In [ ]:
df.show(3, truncate=False)

In [ ]:
df.limit(5).show(10)

## 4.3. Limit i collect

In [ ]:
df.limit(1).collect()

In [ ]:
df.limit(1).collect()[0]

In [ ]:
df.limit(1).collect()[0][1]

## 4.4. Dodawanie kolumny

In [ ]:
df = df.withColumn('SalesX1000', col('sales') * 1000 )

In [ ]:
df.show()

# 5. Kolekcje, daty i funkcje

## 5.1. Lista i słownik w DataFrame

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col, size, lit, explode

In [ ]:
spark = SparkSession.builder.appName("spark").getOrCreate()

In [ ]:
schema = StructType(
    [
        StructField("id", IntegerType(), False),
        StructField("first", StringType(), False),
        StructField("last", StringType(), False),
        StructField("skills", ArrayType(StringType()), False),
        StructField("salary", IntegerType(), False),
        StructField("role", MapType(StringType(), StringType()), False),
        StructField("status", StringType(), True)
    ]
)

In [ ]:
emp = spark.createDataFrame(
    [
        (1, "Adam", "Nowak", ["SQL", "Java", "GCP"], 3500, {"position": "Java Developer", "level": "1"}, None),
        (2, "Jan", "Kowalski", ["SQL", "Java", "Azure", "Spring"], 8000, {"position": "Java Developer", "level": "3"}, "Active"),
        (3, "Dominik", "Bajt", ["Python", "MongoDB", "Redis"], 4000, {"position": "Data Developer", "level": "1"}, None),
        (4, "Ewa", "Piksel", ["SQL", "Python", "Pandas", ], 4100, {"position": "Data Scientist", "level": "1"}, "Fired"),
        (5, "Krzysztof", "Zależność", ["Git", "CI/CD", "Docker"], 8000, {"position": "DevOps", "level": "2"}, "Active"),
        (6, "Ewa", "Kierownik", ["Azure", "GCP", "AWS", "Linux"], 12500, {"position": "Cloud Architect", "level": "2"}, "Fired"),
        (7, "Adam", "Kowalski", ["Git", "CI/CD", "Docker", "Linux", "Kubernetes"], 10500, {"position": "DevOps", "level": "3"}, "New"),
        (8, "Dominika", "Praktyczna", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, None),
        (9, "Jan", "Praktyczny", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, "Active"),
        (10, "Mikołaj", "Sobieski", ["Python", "Django", "Flask"], 7500, {"position": "Python Developer", "level": "1"}, "New")
    ],
    schema
)

In [ ]:
emp.printSchema()

In [ ]:
emp.limit(3).show(truncate=False)

In [ ]:
emp.limit(3).show()

## 5.2. getItem oraz size

In [ ]:
emp.select( col("skills")[1], col("role")["level"] ).show()

In [ ]:
emp.select(col("skills").getItem(1), col("role").getItem('position')).show()

In [ ]:
emp.select(
    col("skills").getItem(1),
    col("role").getItem('position'),
    size(col('skills')),
    size(col('role'))
).show()

## 5.3. lit i explode

In [ ]:
emp.withColumn("company", lit('Dziurex')).show()

In [ ]:
emp.select(
    col('id'),
    explode(col('skills'))
).show()

In [ ]:
emp.select(
    col('id'),
    explode(col('role'))
).show()

## 5.4. Konkatenacja

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DateType, TimestampType
from pyspark.sql.functions import (
    col, size, lit, explode,
    concat, concat_ws, substring,
    datediff, date_add, date_sub,
    year, month, dayofmonth, dayofweek, dayofyear, weekofyear,
    hour, minute, second
)

spark = SparkSession.builder.appName("spark").getOrCreate()

In [ ]:
schema = StructType(
    [
        StructField("id", IntegerType(), False),
        StructField("first", StringType(), False),
        StructField("last", StringType(), False),
        StructField("skills", ArrayType(StringType()), False),
        StructField("salary", IntegerType(), False),
        StructField("role", MapType(StringType(), StringType()), False),
        StructField("status", StringType(), True)
    ]
)

In [ ]:
emp = spark.createDataFrame(
    [
        (1, "Adam", "Nowak", ["SQL", "Java", "GCP"], 3500, {"position": "Java Developer", "level": "1"}, None),
        (2, "Jan", "Kowalski", ["SQL", "Java", "Azure", "Spring"], 8000, {"position": "Java Developer", "level": "3"}, "Active"),
        (3, "Dominik", "Bajt", ["Python", "MongoDB", "Redis"], 4000, {"position": "Data Developer", "level": "1"}, None),
        (4, "Ewa", "Piksel", ["SQL", "Python", "Pandas", ], 4100, {"position": "Data Scientist", "level": "1"}, "Fired"),
        (5, "Krzysztof", "Zależność", ["Git", "CI/CD", "Docker"], 8000, {"position": "DevOps", "level": "2"}, "Active"),
        (6, "Ewa", "Kierownik", ["Azure", "GCP", "AWS", "Linux"], 12500, {"position": "Cloud Architect", "level": "2"}, "Fired"),
        (7, "Adam", "Kowalski", ["Git", "CI/CD", "Docker", "Linux", "Kubernetes"], 10500, {"position": "DevOps", "level": "3"}, "New"),
        (8, "Dominika", "Praktyczna", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, None),
        (9, "Jan", "Praktyczny", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, "Active"),
        (10, "Mikołaj", "Sobieski", ["Python", "Django", "Flask"], 7500, {"position": "Python Developer", "level": "1"}, "New")
    ],
    schema
)

In [ ]:
emp.withColumn("employee", concat(col('first'), lit(' '), col('last'))).show()

In [ ]:
emp.select(
    concat_ws(',', col('id'), col('first'), col('last'))
).show()

## 5.5. substring

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DateType, TimestampType
from pyspark.sql.functions import (
    col, size, lit, explode,
    concat, concat_ws, substring,
    datediff, date_add, date_sub,
    year, month, dayofmonth, dayofweek, dayofyear, weekofyear,
    hour, minute, second
)

spark = SparkSession.builder.appName("spark").getOrCreate()

In [ ]:
schema = StructType(
    [
        StructField("id", IntegerType(), False),
        StructField("first", StringType(), False),
        StructField("last", StringType(), False),
        StructField("skills", ArrayType(StringType()), False),
        StructField("salary", IntegerType(), False),
        StructField("role", MapType(StringType(), StringType()), False),
        StructField("status", StringType(), True)
    ]
)

In [ ]:
emp = spark.createDataFrame(
    [
        (1, "Adam", "Nowak", ["SQL", "Java", "GCP"], 3500, {"position": "Java Developer", "level": "1"}, None),
        (2, "Jan", "Kowalski", ["SQL", "Java", "Azure", "Spring"], 8000, {"position": "Java Developer", "level": "3"}, "Active"),
        (3, "Dominik", "Bajt", ["Python", "MongoDB", "Redis"], 4000, {"position": "Data Developer", "level": "1"}, None),
        (4, "Ewa", "Piksel", ["SQL", "Python", "Pandas", ], 4100, {"position": "Data Scientist", "level": "1"}, "Fired"),
        (5, "Krzysztof", "Zależność", ["Git", "CI/CD", "Docker"], 8000, {"position": "DevOps", "level": "2"}, "Active"),
        (6, "Ewa", "Kierownik", ["Azure", "GCP", "AWS", "Linux"], 12500, {"position": "Cloud Architect", "level": "2"}, "Fired"),
        (7, "Adam", "Kowalski", ["Git", "CI/CD", "Docker", "Linux", "Kubernetes"], 10500, {"position": "DevOps", "level": "3"}, "New"),
        (8, "Dominika", "Praktyczna", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, None),
        (9, "Jan", "Praktyczny", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, "Active"),
        (10, "Mikołaj", "Sobieski", ["Python", "Django", "Flask"], 7500, {"position": "Python Developer", "level": "1"}, "New")
    ],
    schema
)

In [ ]:
emp.select(
    substring(col('first'), 0, 2), # funkcja substring wycina podciąg (fragment tekstu)
    col('first')[0:2] # alternatywna metoda wykorzystująca notację slice
).show()

In [ ]:
emp.select(
    substring(col('first'), 5, 2),
    col('first')[5:2]
).show()

## 5.6. DateType i TimestampType

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DateType, TimestampType
from pyspark.sql.functions import (
    col, size, lit, explode,
    concat, concat_ws, substring,
    datediff, date_add, date_sub,
    year, month, dayofmonth, dayofweek, dayofyear, weekofyear,
    hour, minute, second
)

from datetime import datetime

spark = SparkSession.builder.appName("spark").getOrCreate()

In [ ]:
schema = StructType(
    [
        StructField("id", IntegerType(), False),
        StructField("first", StringType(), False),
        StructField("last", StringType(), False),
        StructField("skills", ArrayType(StringType()), False),
        StructField("salary", IntegerType(), False),
        StructField("role", MapType(StringType(), StringType()), False),
        StructField("status", StringType(), True),
        StructField("hire_date", DateType(), True),
        StructField("hire_timestamp", TimestampType(), True)
    ]
)

emp = spark.createDataFrame(
    [
        (1, "Adam", "Nowak", ["SQL", "Java", "GCP"], 3500, {"position": "Java Developer", "level": "1"}, None,
         datetime(2023, 5, 1), datetime(2023, 5, 1, 12, 0, 0)),
        (2, "Jan", "Kowalski", ["SQL", "Java", "Azure", "Spring"], 8000, {"position": "Java Developer", "level": "3"}, "Active",
         datetime(2023, 5, 10), datetime(2023, 5, 10, 16, 0, 0)),
        (3, "Dominik", "Bajt", ["Python", "MongoDB", "Redis"], 4000, {"position": "Data Developer", "level": "1"}, None,
         datetime(2023, 5, 15), datetime(2023, 5, 15, 8, 0, 0)),
        (4, "Ewa", "Piksel", ["SQL", "Python", "Pandas", ], 4100, {"position": "Data Scientist", "level": "1"}, "Fired",
         datetime(2023, 6, 10), datetime(2023, 6, 1, 11, 0, 0)),
        (5, "Krzysztof", "Zależność", ["Git", "CI/CD", "Docker"], 8000, {"position": "DevOps", "level": "2"}, "Active",
         datetime(2023, 6, 15), datetime(2023, 6, 15, 11, 30, 0)),
        (6, "Ewa", "Kierownik", ["Azure", "GCP", "AWS", "Linux"], 12500, {"position": "Cloud Architect", "level": "2"}, "Fired",
         datetime(2023, 6, 20), datetime(2023, 6, 20, 12, 0)),
        (7, "Adam", "Kowalski", ["Git", "CI/CD", "Docker", "Linux", "Kubernetes"], 10500, {"position": "DevOps", "level": "3"}, "New",
         datetime(2023, 1, 20), datetime(2023, 1, 20, 9, 0)),
        (8, "Dominika", "Praktyczna", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, None,
         datetime(2023, 1, 30), datetime(2023, 1, 30, 7, 0)),
        (9, "Jan", "Praktyczny", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, "Active",
         datetime(2023, 3, 20), datetime(2023, 3, 20, 11, 45)),
        (10, "Mikołaj", "Sobieski", ["Python", "Django", "Flask"], 7500, {"position": "Python Developer", "level": "1"}, "New",
         datetime(2023, 1, 20), datetime(2023, 1, 20, 8, 35))
    ],
    schema
)

In [ ]:
emp.show()

## 5.7. datediff

In [ ]:
emp.select(
    datediff(col('hire_date'), lit(datetime(2023, 9, 1))),
    datediff(lit(datetime(2023, 9, 1)), col('hire_date'))
).show()

## 5.8. date_add/date_sub

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DateType, TimestampType
from pyspark.sql.functions import (
    col, size, lit, explode,
    concat, concat_ws, substring,
    datediff, date_add, date_sub,
    year, month, dayofmonth, dayofweek, dayofyear, weekofyear,
    hour, minute, second
)

from datetime import datetime

spark = SparkSession.builder.appName("spark").getOrCreate()

In [ ]:
schema = StructType(
    [
        StructField("id", IntegerType(), False),
        StructField("first", StringType(), False),
        StructField("last", StringType(), False),
        StructField("skills", ArrayType(StringType()), False),
        StructField("salary", IntegerType(), False),
        StructField("role", MapType(StringType(), StringType()), False),
        StructField("status", StringType(), True),
        StructField("hire_date", DateType(), True),
        StructField("hire_timestamp", TimestampType(), True)
    ]
)

emp = spark.createDataFrame(
    [
        (1, "Adam", "Nowak", ["SQL", "Java", "GCP"], 3500, {"position": "Java Developer", "level": "1"}, None,
         datetime(2023, 5, 1), datetime(2023, 5, 1, 12, 0, 0)),
        (2, "Jan", "Kowalski", ["SQL", "Java", "Azure", "Spring"], 8000, {"position": "Java Developer", "level": "3"}, "Active",
         datetime(2023, 5, 10), datetime(2023, 5, 10, 16, 0, 0)),
        (3, "Dominik", "Bajt", ["Python", "MongoDB", "Redis"], 4000, {"position": "Data Developer", "level": "1"}, None,
         datetime(2023, 5, 15), datetime(2023, 5, 15, 8, 0, 0)),
        (4, "Ewa", "Piksel", ["SQL", "Python", "Pandas", ], 4100, {"position": "Data Scientist", "level": "1"}, "Fired",
         datetime(2023, 6, 10), datetime(2023, 6, 1, 11, 0, 0)),
        (5, "Krzysztof", "Zależność", ["Git", "CI/CD", "Docker"], 8000, {"position": "DevOps", "level": "2"}, "Active",
         datetime(2023, 6, 15), datetime(2023, 6, 15, 11, 30, 0)),
        (6, "Ewa", "Kierownik", ["Azure", "GCP", "AWS", "Linux"], 12500, {"position": "Cloud Architect", "level": "2"}, "Fired",
         datetime(2023, 6, 20), datetime(2023, 6, 20, 12, 0)),
        (7, "Adam", "Kowalski", ["Git", "CI/CD", "Docker", "Linux", "Kubernetes"], 10500, {"position": "DevOps", "level": "3"}, "New",
         datetime(2023, 1, 20), datetime(2023, 1, 20, 9, 0)),
        (8, "Dominika", "Praktyczna", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, None,
         datetime(2023, 1, 30), datetime(2023, 1, 30, 7, 0)),
        (9, "Jan", "Praktyczny", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, "Active",
         datetime(2023, 3, 20), datetime(2023, 3, 20, 11, 45)),
        (10, "Mikołaj", "Sobieski", ["Python", "Django", "Flask"], 7500, {"position": "Python Developer", "level": "1"}, "New",
         datetime(2023, 1, 20), datetime(2023, 1, 20, 8, 35))
    ],
    schema
)

In [ ]:
emp.select(
    col('hire_date'),
    date_add(col('hire_date'), 10),
    date_sub(col('hire_date'), 30)
).show()

In [ ]:
emp.select(
    col('hire_date'),
    date_add(col('hire_date'), -10),
    date_sub(col('hire_date'), -30)
).show()

## 5.9. Ekstrakcja danej jednostki czasu

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DateType, TimestampType
from pyspark.sql.functions import (
    col, size, lit, explode,
    concat, concat_ws, substring,
    datediff, date_add, date_sub,
    year, month, dayofmonth, dayofweek, dayofyear, weekofyear,
    hour, minute, second
)

from datetime import datetime

spark = SparkSession.builder.appName("spark").getOrCreate()

In [ ]:
schema = StructType(
    [
        StructField("id", IntegerType(), False),
        StructField("first", StringType(), False),
        StructField("last", StringType(), False),
        StructField("skills", ArrayType(StringType()), False),
        StructField("salary", IntegerType(), False),
        StructField("role", MapType(StringType(), StringType()), False),
        StructField("status", StringType(), True),
        StructField("hire_date", DateType(), True),
        StructField("hire_timestamp", TimestampType(), True)
    ]
)

emp = spark.createDataFrame(
    [
        (1, "Adam", "Nowak", ["SQL", "Java", "GCP"], 3500, {"position": "Java Developer", "level": "1"}, None,
         datetime(2023, 5, 1), datetime(2023, 5, 1, 12, 0, 0)),
        (2, "Jan", "Kowalski", ["SQL", "Java", "Azure", "Spring"], 8000, {"position": "Java Developer", "level": "3"}, "Active",
         datetime(2023, 5, 10), datetime(2023, 5, 10, 16, 0, 0)),
        (3, "Dominik", "Bajt", ["Python", "MongoDB", "Redis"], 4000, {"position": "Data Developer", "level": "1"}, None,
         datetime(2023, 5, 15), datetime(2023, 5, 15, 8, 0, 0)),
        (4, "Ewa", "Piksel", ["SQL", "Python", "Pandas", ], 4100, {"position": "Data Scientist", "level": "1"}, "Fired",
         datetime(2023, 6, 10), datetime(2023, 6, 1, 11, 0, 0)),
        (5, "Krzysztof", "Zależność", ["Git", "CI/CD", "Docker"], 8000, {"position": "DevOps", "level": "2"}, "Active",
         datetime(2023, 6, 15), datetime(2023, 6, 15, 11, 30, 0)),
        (6, "Ewa", "Kierownik", ["Azure", "GCP", "AWS", "Linux"], 12500, {"position": "Cloud Architect", "level": "2"}, "Fired",
         datetime(2023, 6, 20), datetime(2023, 6, 20, 12, 0)),
        (7, "Adam", "Kowalski", ["Git", "CI/CD", "Docker", "Linux", "Kubernetes"], 10500, {"position": "DevOps", "level": "3"}, "New",
         datetime(2023, 1, 20), datetime(2023, 1, 20, 9, 0)),
        (8, "Dominika", "Praktyczna", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, None,
         datetime(2023, 1, 30), datetime(2023, 1, 30, 7, 0)),
        (9, "Jan", "Praktyczny", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, "Active",
         datetime(2023, 3, 20), datetime(2023, 3, 20, 11, 45)),
        (10, "Mikołaj", "Sobieski", ["Python", "Django", "Flask"], 7500, {"position": "Python Developer", "level": "1"}, "New",
         datetime(2023, 1, 20), datetime(2023, 1, 20, 8, 35))
    ],
    schema
)

In [ ]:
(
    emp
    .withColumn( 'year', year(col('hire_timestamp')) )
    .withColumn( 'day_of_week', dayofweek(col('hire_timestamp')) )
    .withColumn( 'week_of_year', weekofyear(col('hire_timestamp')) )
    .withColumn( 'hour', hour(col('hire_timestamp')) )
    .limit(3)
    .show()
)

# 6. Filtrowanie danych

## 6.1. Unikatowe wiersze

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DateType, TimestampType
from pyspark.sql.functions import (
    col, size, lit, explode,
    concat, concat_ws, substring,
    datediff, date_add, date_sub,
    year, month, dayofmonth, dayofweek, dayofyear, weekofyear,
    hour, minute, second
)

from datetime import datetime

spark = SparkSession.builder.appName("spark").getOrCreate()

In [ ]:
schema = StructType(
    [
        StructField("id", IntegerType(), False),
        StructField("first", StringType(), False),
        StructField("last", StringType(), False),
        StructField("skills", ArrayType(StringType()), False),
        StructField("salary", IntegerType(), False),
        StructField("role", MapType(StringType(), StringType()), False),
        StructField("status", StringType(), True),
        StructField("hire_date", DateType(), True),
        StructField("hire_timestamp", TimestampType(), True)
    ]
)

emp = spark.createDataFrame(
    [
        (1, "Adam", "Nowak", ["SQL", "Java", "GCP"], 3500, {"position": "Java Developer", "level": "1"}, None,
         datetime(2023, 5, 1), datetime(2023, 5, 1, 12, 0, 0)),
        (2, "Jan", "Kowalski", ["SQL", "Java", "Azure", "Spring"], 8000, {"position": "Java Developer", "level": "3"}, "Active",
         datetime(2023, 5, 10), datetime(2023, 5, 10, 16, 0, 0)),
        (3, "Dominik", "Bajt", ["Python", "MongoDB", "Redis"], 4000, {"position": "Data Developer", "level": "1"}, None,
         datetime(2023, 5, 15), datetime(2023, 5, 15, 8, 0, 0)),
        (4, "Ewa", "Piksel", ["SQL", "Python", "Pandas", ], 4100, {"position": "Data Scientist", "level": "1"}, "Fired",
         datetime(2023, 6, 10), datetime(2023, 6, 1, 11, 0, 0)),
        (5, "Krzysztof", "Zależność", ["Git", "CI/CD", "Docker"], 8000, {"position": "DevOps", "level": "2"}, "Active",
         datetime(2023, 6, 15), datetime(2023, 6, 15, 11, 30, 0)),
        (6, "Ewa", "Kierownik", ["Azure", "GCP", "AWS", "Linux"], 12500, {"position": "Cloud Architect", "level": "2"}, "Fired",
         datetime(2023, 6, 20), datetime(2023, 6, 20, 12, 0)),
        (7, "Adam", "Kowalski", ["Git", "CI/CD", "Docker", "Linux", "Kubernetes"], 10500, {"position": "DevOps", "level": "3"}, "New",
         datetime(2023, 1, 20), datetime(2023, 1, 20, 9, 0)),
        (8, "Dominika", "Praktyczna", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, None,
         datetime(2023, 1, 30), datetime(2023, 1, 30, 7, 0)),
        (9, "Jan", "Praktyczny", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, "Active",
         datetime(2023, 3, 20), datetime(2023, 3, 20, 11, 45)),
        (10, "Mikołaj", "Sobieski", ["Python", "Django", "Flask"], 7500, {"position": "Python Developer", "level": "1"}, "New",
         datetime(2023, 1, 20), datetime(2023, 1, 20, 8, 35))
    ],
    schema
)

In [ ]:
emp.printSchema()

In [ ]:
games = spark.read.csv('data/Games.csv', header=True).select(col('Series'), col('Developer'), col('Publisher')).orderBy(col('Developer'))

In [ ]:
games.limit(10).show(truncate=False)

In [ ]:
games.distinct().orderBy(col('Developer')).limit(10).show(truncate=False)

In [ ]:
games.dropDuplicates().orderBy(col('Developer')).limit(10).show(truncate=False)

In [ ]:
games.dropDuplicates(['Series']).orderBy(col('Developer')).limit(10).show(truncate=False)

## 6.2. Filtrowanie danych cz. 1

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DateType, TimestampType
from pyspark.sql.functions import (
    col, size, lit, explode,
    concat, concat_ws, substring,
    datediff, date_add, date_sub,
    year, month, dayofmonth, dayofweek, dayofyear, weekofyear,
    hour, minute, second
)

from datetime import datetime

spark = SparkSession.builder.appName("spark").getOrCreate()

In [ ]:
books = spark.read.csv('data/best_selling_books.csv', header=True)

In [ ]:
books.limit(5).show()

In [ ]:
books.filter(
    col('Original language') == 'Portuguese'

).show()

In [ ]:
books.filter(
    col('First published') > 2010

).show(truncate=False)

In [ ]:
emp.limit(5).show()

Zapis funkcyjny

In [ ]:
emp.filter(
    size(col('skills')) > 3
).show(truncate=False)

Zapis SQL

In [ ]:
emp.filter(
    "last == 'Kowalski' "
).show()

In [ ]:
emp.filter(
    "hire_date > '2023-05-10' "
).show()

In [ ]:
emp.filter(
    col('hire_date') > datetime(2023, 5, 10)
).show()

## 6.3. Filtrowanie danych cz. 2

In [ ]:
emp.filter(
    col('role').getItem('position').isin(['DevOps', 'Intern'])
).show(truncate=False)

In [ ]:
emp.filter(
    col('salary').between(3000, 4000)
).show()

In [ ]:
emp.filter(
    "salary between 3000 and 4000"
).show()

In [ ]:
emp.filter(
    col('status').isNull()
).show()

In [ ]:
emp.filter(
    ~col('status').isNull()
).show()

## 6.4. Łączenie warunków

In [ ]:
books = spark.read.csv('data/best_selling_books.csv', header=True)

In [ ]:
books.limit(3).show(truncate=False)

In [ ]:
books.filter(
    (col('First published') < 2000 )
    &
    (col('Original language') != 'English')
).limit(5).show()

In [ ]:
books.filter(
    (col('Genre') == 'Detective')
    |
    (col('First published') > 2000)
).limit(5).show()


In [ ]:
books.filter(
    ( (col('Genre') == 'Detective')
    |
    (col('First published') > 2000) )
    &
    (col('Original language') == 'Hindi')
).limit(5).show()

# 7. Grupowanie danych

## 7.1. Funkcje agregujące/alias

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DateType, TimestampType
from pyspark.sql.functions import (
    col, size, lit, explode,
    concat, concat_ws, substring,
    datediff, date_add, date_sub,
    year, month, dayofmonth, dayofweek, dayofyear, weekofyear,
    hour, minute, second,
    count, min, max, avg, sum
)
from datetime import datetime

In [ ]:
spark = SparkSession.builder.appName("spark").getOrCreate()

In [ ]:
schema = StructType(
    [
        StructField("id", IntegerType(), False),
        StructField("first", StringType(), False),
        StructField("last", StringType(), False),
        StructField("skills", ArrayType(StringType()), False),
        StructField("salary", IntegerType(), False),
        StructField("role", MapType(StringType(), StringType()), False),
        StructField("status", StringType(), True),
        StructField("hire_date", DateType(), True),
        StructField("hire_timestamp", TimestampType(), True),
        StructField("country_code", StringType(), True)
    ]
)

emp = spark.createDataFrame(
    [
        (1, "Adam", "Nowak", ["SQL", "Java", "GCP"], 3500, {"position": "Java Developer", "level": "1"}, None,
         datetime(2023, 5, 1), datetime(2023, 5, 1, 12, 0, 0), "PL"),
        (2, "Jan", "Kowalski", ["SQL", "Java", "Azure", "Spring"], 8000, {"position": "Java Developer", "level": "3"}, "Active",
         datetime(2023, 5, 10), datetime(2023, 5, 1, 16, 0, 0), "PL"),
        (3, "Dominik", "Bajt", ["Python", "MongoDB", "Redis"], 4000, {"position": "Data Developer", "level": "1"}, None,
         datetime(2023, 5, 15), datetime(2023, 5, 15, 8, 0, 0), "GB"),
        (4, "Ewa", "Piksel", ["SQL", "Python", "Pandas", ], 4100, {"position": "Data Scientist", "level": "1"}, "Fired",
         datetime(2023, 6, 10), datetime(2023, 6, 1, 11, 0, 0), "DE"),
        (5, "Krzysztof", "Zależność", ["Git", "CI/CD", "Docker"], 8000, {"position": "DevOps", "level": "2"}, "Active",
         datetime(2023, 6, 15), datetime(2023, 6, 15, 11, 30, 0), "DE"),
        (6, "Ewa", "Kierownik", ["Azure", "GCP", "AWS", "Linux"], 12500, {"position": "Cloud Architect", "level": "2"}, "Fired",
         datetime(2023, 6, 20), datetime(2023, 6, 20, 12, 0), "CZ"),
        (7, "Adam", "Kowalski", ["Git", "CI/CD", "Docker", "Linux", "Kubernetes"], 10500, {"position": "DevOps", "level": "3"}, "New",
         datetime(2023, 1, 20), datetime(2023, 1, 20, 9, 0), "CZ"),
        (8, "Dominika", "Praktyczna", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, None,
         datetime(2023, 1, 30), datetime(2023, 1, 30, 7, 0), "GB"),
        (9, "Jan", "Praktyczny", ["SQL", "Java", "Python"], 3000, {"position": "Intern", "level": "0"}, "Active",
         datetime(2023, 3, 20), datetime(2023, 3, 20, 11, 45), "AT"),
        (10, "Mikołaj", "Sobieski", ["Python", "Django", "Flask"], 7500, {"position": "Python Developer", "level": "1"}, "New",
         datetime(2023, 1, 20), datetime(2023, 1, 20, 8, 35), "AT")
    ],
    schema
)

countries_schema = StructType(
    [
    StructField("country", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country_code_three", StringType(), True)
    ]
)

In [ ]:
countries = spark.read.csv("data/country-codes.csv", header=False, sep=";", schema=countries_schema)

countries.limit(5).show(truncate=False)

In [ ]:
emp.limit(5).show()

In [ ]:
countries.agg(
    count("*")
).show()

In [ ]:
countries.agg(
    count("*").alias('liczba wierszy')
).show()

In [ ]:
emp.agg(
    sum(col('salary')).alias('wynagrodzenie').alias('sum solary'),
    max(col('salary')).alias('wynagrodzenie').alias('max solary'),
    min(col('salary')).alias('wynagrodzenie').alias('min solary'),
    avg(col('salary')).alias('wynagrodzenie').alias('avg solary')

).show()

## 7.2. Grupowanie danych

In [ ]:
emp.groupBy(
    col('role').getItem('position')
).agg(
    avg(col('salary'))
).show()

In [ ]:
emp.groupBy(
    col('role').getItem('position'),
    col('first')
).agg(
    avg(col('salary'))
).show()

In [ ]:
emp.limit(3).show()

In [ ]:
emp.withColumn('skill', explode(col('skills'))).limit(5).show()

In [ ]:
(
    emp
    .withColumn('skill', explode(col('skills')))
    .groupBy(col('skill'))
    .agg(count('*'))
    .limit(10).show()
)

## 7.3. JOIN

In [ ]:
emp.show(2)
countries.show(2)

In [ ]:
emp_and_country = emp.join(
    countries,
    emp.country_code == countries.country_code,
    'inner'
)

In [ ]:
emp_and_country.limit(5).show()

In [ ]:
emp_and_country = emp.join(
    countries,
    emp["country_code"] == countries["country_code"],
    "inner"
)

emp_and_country.show()

In [ ]:
emp_and_country = emp.join(
    countries,
    on='country_code',
    how='inner'
)

emp_and_country.show()

## 7.4. Union/UnionAll

In [ ]:
d1 = spark.read.csv("data/Games.csv", header=True).select(
    lit("Games").alias("Type"), col("Name"), col("Developer")
).orderBy(col("Developer")).limit(10)

d2 = spark.read.csv("data/best_selling_books.csv", header=True).select(
    lit("Book").alias("Type"), col("Book"), col("Author(s)")
).limit(10)

In [ ]:
d1.show()
d2.show()

In [ ]:
df3 = d1.union(d2)
df3.show()

In [ ]:
df4 = d1.unionAll(d2)
df4.show()

# 8. Mapowanie i funkcje użytkownika

## 8.1. Funkcje użytkownika UDF

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DateType, TimestampType
from pyspark.sql.functions import (
    col, size, lit, explode,
    concat, concat_ws, substring,
    datediff, date_add, date_sub,
    year, month, dayofmonth, dayofweek, dayofyear, weekofyear,
    hour, minute, second,
    count, min, max, avg, sum, udf, when
)
from datetime import datetime

In [ ]:
spark = SparkSession.builder.appName("spark").getOrCreate()

In [ ]:
games = spark.read.csv("data/Games.csv", header=True).select(col("Series"), col("Developer"), col("Publisher"))

In [ ]:
games.limit(5).show(truncate=False)

In [ ]:
def to_upper(input_str: str) -> str:
    return input_str.upper()

In [ ]:
upper_case_udf = udf(
    lambda x: to_upper(x),
    StringType()
)

In [ ]:
games.withColumn('upper_developer', upper_case_udf(col('Developer'))).show()

## 8.2. Funkcja when

In [ ]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DateType, TimestampType
from pyspark.sql.functions import (
    col, size, lit, explode,
    concat, concat_ws, substring,
    datediff, date_add, date_sub,
    year, month, dayofmonth, dayofweek, dayofyear, weekofyear,
    hour, minute, second,
    count, min, max, avg, sum, udf, when
)
from datetime import datetime

In [ ]:
spark = SparkSession.builder.appName("spark").getOrCreate()

In [ ]:
schema = StructType(
    [
        StructField("first", StringType(), False),
        StructField("last", StringType(), False),
        StructField("salary", IntegerType(), False),
    ]
)

emp = spark.createDataFrame(
    [
        ("Adam", "Nowak", 3500),
        ("Jan", "Kowalski", 8000),
        ("Dominik", "Bajt", 4000),
        ("Ewa", "Piksel", 4100),
        ("Krzysztof", "Zależność", 8000),
        ("Ewa", "Kierownik", 12500),
        ("Adam", "Kowalski", 10500),
        ("Dominika", "Praktyczna", 3000),
        ("Jan", "Praktyczny", 3000),
        ("Mikołaj", "Sobieski", 7500)
    ],
    schema
)

In [ ]:
emp.show()

In [ ]:
(
    emp.withColumn('salary_tier',
                   when(col('salary') >= 10000, 'I Tier')
                   .when(col('salary') >= 8000, 'II Tier')
                   .otherwise('III Tier')
                  )
).show()

## 8.3. Funkcja map

In [ ]:
emp.show()

In [ ]:
new_schema = StructType(
    [
        StructField('body', MapType(StringType(), StringType()), False)
    ]
)

In [ ]:
def map_to_json(r):
    return Row(
        body={
            'first': r.first,
            'last': r.last,
            'year_salary': str(r.salary * 12)
        }
    )

In [ ]:
body_emp = emp.rdd.map(lambda r: map_to_json(r)).toDF(new_schema)
body_emp.show(truncate=False)

## 8.4. Funkcja flatMap

In [ ]:
def map_to_json_with_currency(r):
    return [
        Row(body={ 'first': r.first, 'last': r.last, 'year_salary': str(r.salary * 12), 'currency': 'PLN' }),
        Row(body={ 'first': r.first, 'last': r.last, 'year_salary': str(r.salary * 12 / 4.35), 'currency': 'EUR' })
    ]

In [ ]:
body_emp = emp.rdd.flatMap(lambda r: map_to_json_with_currency(r)).toDF(new_schema)
body_emp.show(truncate=False)

# 9. Zapisywanie danych do pliku

## 9.1. Omówienie formatów danych

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, MapType, DateType, TimestampType

In [ ]:
spark = SparkSession.builder.appName("spark").getOrCreate()

#### Formaty danych (najpopularniejsze dla PySpark)

1. csv - czytelny dla użytkownika, prosty w implementacji, zapis w postaci łańcuchów znakowych
2. json - zapis klucz:wartość, popularny format JSON,
3. parquet - format kolumnowy, "write once, read many", zapis binarny, przechowuje również schemat danych
4. avro - format o wysokiej kompresji, dobry do archiwizacji danych, zapis binarny, przechowuje schemat

## 9.2. Zapis do pliku

In [ ]:
schema = StructType(
    [
        StructField("first", StringType(), False),
        StructField("last", StringType(), False),
        StructField("salary", IntegerType(), False),
    ]
)

emp = spark.createDataFrame(
    [
        ("Adam", "Nowak", 3500),
        ("Jan", "Kowalski", 8000),
        ("Dominik", "Bajt", 4000),
        ("Ewa", "Piksel", 4100),
        ("Krzysztof", "Zależność", 8000),
        ("Ewa", "Kierownik", 12500),
        ("Adam", "Kowalski", 10500),
        ("Dominika", "Praktyczna", 3000),
        ("Jan", "Praktyczny", 3000),
        ("Mikołaj", "Sobieski", 7500)
    ],
    schema
)

In [ ]:
emp.write.mode("overwrite").parquet("data/emp.parquet")

In [ ]:
spark.read.csv('data/import', header=True).show()

## 9.3. Spark SQL

In [ ]:
emp.createOrReplaceTempView('widok')

In [ ]:
spark.sql(
    "SELECT first, count(*) as cnt FROM widok GROUP BY first order by cnt desc"
).show()

In [ ]:
spark.sql(
    "SELECT * FROM widok"
).show()